# Movies & Shows Data Analysis

**Author:** Danisha L. Thomas, PhD  
**Program:** TripleTen Data Science Bootcamp — Sprint 1  
**Tools:** Python, Pandas

---

## Project Overview

This project applies foundational data analysis skills using the Pandas library to explore a dataset of movies and television shows. The analysis covers data loading, column standardization, data cleaning, filtering, and the creation of reusable functions for querying the dataset.

## Dataset Description

The dataset `movies_and_shows.csv` contains cast and title information sourced from IMDb, including:

| Column | Description |
|--------|-------------|
| `name` | Actor or actress name |
| `character` | Character portrayed |
| `role` | Role type (e.g., ACTOR) |
| `title` | Movie or show title |
| `type` | MOVIE or SHOW |
| `release_year` | Year of release |
| `genres` | List of genres |
| `imdb_score` | IMDb rating |
| `imdb_votes` | Number of IMDb votes |

## Setup: Import Libraries and Load Data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/datasets/movies_and_shows.csv')
print(df.head())

              name                Character   r0le        TITLE   Type  \
0   Robert De Niro            Travis Bickle  ACTOR  Taxi Driver  MOVIE   
1     Jodie Foster            Iris Steensma  ACTOR  Taxi Driver  MOVIE   
2    Albert Brooks                      Tom  ACTOR  Taxi Driver  MOVIE   
3    Harvey Keitel  Matthew 'Sport' Higgins  ACTOR  Taxi Driver  MOVIE   
4  Cybill Shepherd                    Betsy  ACTOR  Taxi Driver  MOVIE   

   release Year              genres  imdb sc0re  imdb v0tes  
0          1976  ['drama', 'crime']         8.2    808582.0  
1          1976  ['drama', 'crime']         8.2    808582.0  
2          1976  ['drama', 'crime']         8.2    808582.0  
3          1976  ['drama', 'crime']         8.2    808582.0  
4          1976  ['drama', 'crime']         8.2    808582.0  


## Task 1: Data Cleaning — Standardize Column Names

The raw dataset contains inconsistent column names with typos, mixed casing, spaces, and zero-substituted characters. The `.rename()` method is used to standardize all column names to lowercase snake_case for consistency and ease of use throughout the analysis.

In [ ]:
df.info()

In [ ]:
df = df.rename(columns=str.strip).rename(columns={
    '  name ': 'name',
    'Character': 'character',
    'r0le': 'role',
    'TITLE': 'title',
    'Type': 'type',
    'release Year': 'release_year',
    'imdb sc0re': 'imdb_score',
    'imdb v0tes': 'imdb_votes',
})
print(df.columns.tolist())

## Task 2: Correct a Misspelled Actor Name

The dataset contains an encoding error at index 85576 where the actor's name appears as `"In??s Prieto"`. Using `.loc[]`, the incorrect entry is identified, corrected to `"Ines Prieto"`, and verified.

In [ ]:
# Locate the row with the encoding error
print(df.loc[85576])

In [ ]:
# Apply the correction
df.loc[85576, 'name'] = 'Ines Prieto'

In [ ]:
# Verify the correction was applied
print(df.loc[85576])

## Task 3: Filter All Titles Featuring Ines Prieto

With the name corrected, we can now accurately retrieve all movies and shows featuring Ines Prieto. The DataFrame is filtered on the `name` column and relevant columns are selected for a clean output.

In [ ]:
result = df[df['name'] == 'Ines Prieto'][['title', 'release_year', 'genres', 'imdb_score']]
print(result)

## Task 4: Identify Highly Rated Titles (IMDb Score > 9.0)

To surface the highest-rated content in the dataset, we filter for titles with an IMDb score greater than 9.0 and extract a unique set of titles to eliminate duplicates caused by multiple cast rows per title.

In [ ]:
imdb_score_df = df[df['imdb_score'] > 9]
title = imdb_score_df['title']
unique_title = set(title)
print(unique_title)

## Task 5: Function — Get Unique Top-Rated Titles by Minimum Score

This reusable function accepts a minimum IMDb score as input and returns a set of unique titles that meet or exceed that threshold. Using a `set()` ensures no duplicates appear in the output regardless of how many cast members are listed per title.

In [ ]:
def get_unique_top_movies(min_score):
    high_score_df = df[df['imdb_score'] >= min_score]
    high_score_titles = high_score_df['title']
    unique_titles = set(high_score_titles)
    return unique_titles

# Test: titles with IMDb score >= 9.0
print(get_unique_top_movies(9.0))

## Task 6: Function — Get Top-Rated Titles from a Specific Decade

This function takes a decade start year and a minimum IMDb score, then returns unique titles released within that decade that meet the score threshold. It combines two filter conditions using `&` for compound Boolean indexing.

In [ ]:
def get_top_movies_from_decade(decade_start, min_score):
    decade_df = df[
        (df['release_year'] >= decade_start) &
        (df['release_year'] <= decade_start + 9)
    ]
    high_score_decade_df = decade_df[decade_df['imdb_score'] >= min_score]
    high_score_decade = high_score_decade_df['title']
    unique_titles = set(high_score_decade)
    return unique_titles

# Test: top-rated titles from the 1990s with IMDb score >= 8.5
print(get_top_movies_from_decade(1990, 8.5))

## Task 7: Function — List All Actors for a Given Title

This function accepts a title string and returns a comma-separated string of all actors credited in that title. It filters on both the `title` and `role` columns to ensure only actors (not directors or other roles) are included.

In [ ]:
def get_actors_for_title(title):
    actors_df = df[(df['title'] == title) & (df['role'] == 'ACTOR')]
    names = actors_df['name']
    names_string = ', '.join(names)
    return names_string

# Test: actors in Taxi Driver
print(get_actors_for_title("Taxi Driver"))

## Task 8: Function — Categorize Titles by IMDb Score

This function classifies any title into one of four quality tiers based on its IMDb score. If the title is not found in the dataset, a `"Title not found"` message is returned. The scoring rubric is as follows:

| Category | IMDb Score Range |
|----------|------------------|
| Excellent | ≥ 9.0 |
| Good | 7.0 – 8.9 |
| Average | 5.0 – 6.9 |
| Low | < 5.0 |

In [ ]:
def categorize_imdb_score(title):
    titles_df = df[df['title'] == title]
    
    if len(titles_df) == 0:
        return "Title not found"

    score = titles_df['imdb_score'].iloc[0]

    if score >= 9.0:
        return "Excellent"
    elif score >= 7.0:
        return "Good"
    elif score >= 5.0:
        return "Average"
    else:
        return "Low"

# Test: categorize Taxi Driver
print(categorize_imdb_score("Taxi Driver"))

---

## Summary

This project demonstrated core Pandas skills applied to a real-world entertainment dataset:

- **Data Loading & Inspection** — loaded the dataset and reviewed structure using `.head()` and `.info()`
- **Column Standardization** — corrected inconsistent naming conventions using `.rename()`
- **Data Correction** — identified and fixed an encoding error using `.loc[]`
- **Filtering & Boolean Indexing** — queried the dataset using single and compound conditions
- **Reusable Functions** — built four modular functions for flexible querying by score, decade, title, and actor

These foundational techniques form the basis for all subsequent exploratory and statistical analysis work in this program.